# 03 — Modelagem Econômica por Alavanca e Auditoria contra a Base Real

**Objetivo:** Definir as premissas explícitas por alavanca (canibalização, elasticidade e custo operacional/SLA), calcular pedidos incrementais líquidos, GMV incremental e MC incremental, e auditar **AOV, taxa de MC, market share, saturação e Price Index** contra a base real.


In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

PROJECT_DIR = "/content/drive/MyDrive/ifood-growth-competition-case"
if os.path.exists("/content/drive"):
    if not os.path.exists(PROJECT_DIR):
        from google.colab import drive
        drive.mount("/content/drive")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        PROJECT_DIR = os.getcwd()

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

for sub in ["data/raw", "data/processed", "outputs/excel", "outputs/figures", "outputs/reports", "dashboard"]:
    os.makedirs(sub, exist_ok=True)

print("Diretório ativo:", os.getcwd())


In [ ]:
def load_or_build_base():
    candidates = [
        "Case_Competição_prático - Dados.csv",
        "data/raw/Case_Competição_prático - Dados.csv",
        "data/processed/base_validada.csv",
    ]
    raw_data = [
        ("City 1",  329465.76,  22126920, 67, 58,  71.38, 50.19,  1076, 8.16,  -0.01, 93.0,  6.0, 8.0, 56,  91.07),
        ("City 10",  47493.57,   3343072, 70, 60, 168.30, 56.53,    58, 7.49,  -3.58, 83.0,  9.0, 5.0, 38,  97.68),
        ("City 11", 258969.66,  16457522, 64, 57, 147.87, 63.27,   324, 7.78,   6.21, 73.0,  9.0, 5.0, 49, 113.08),
        ("City 12", 175600.89,  11507126, 66, 56,  87.94, 61.73,   378, 8.66,   1.88, 93.0, 10.0, 8.0, 44,  94.74),
        ("City 13", 316311.15,  21844448, 69, 60,  70.38, 53.76,   978, 8.72,   2.84, 86.0, 14.0, 4.0, 41, 101.32),
        ("City 14",  45408.99,   3442456, 76, 65,  96.14, 62.83,    88, 8.48,   0.89, 95.0, 15.0, 7.0, 48,  94.85),
        ("City 15", 562253.85,  39211583, 70, 62,  86.18, 56.72,  1346, 7.29,   0.07, 91.0, 11.0, 7.0, 29,  95.50),
        ("City 16", 308942.97,  21953487, 71, 61,  82.17, 46.77,   941, 7.53,   1.62, 86.0, 15.0, 6.0, 20,  93.75),
        ("City 17",2003389.05, 162615089, 81, 73, 131.38, 57.83,  3085, 7.38,  -1.34, 67.0,  9.0, 8.0, 54, 108.99),
        ("City 18", 630705.33,  41916676, 66, 59,  78.48, 45.01,  2089, 7.72,  -4.69, 65.0,  5.0, 7.0, 54,  93.49),
        ("City 19",  37382.58,   2842198, 76, 69,  98.82, 55.28,    80, 7.07,   6.46, 65.0,  6.0, 7.0, 49,  97.07),
        ("City 2", 1213681.77,  87773466, 72, 63,  83.86, 60.18,  2814, 8.96,   8.41, 67.0, 13.0, 5.0, 55, 116.84),
        ("City 20",7443818.73, 635776558, 85, 78, 141.21, 47.54, 12974, 8.03,   4.78, 74.0, 13.0, 8.0, 46, 114.44),
        ("City 3", 1318842.06, 101207940, 77, 69,  79.81, 40.71,  4749, 8.22,   6.06, 76.0,  6.0, 7.0, 27, 101.27),
        ("City 4",  793354.74,  52163074, 66, 57, 129.05, 54.51,  1319, 7.78,   3.39, 73.0,  5.0, 4.0, 59,  96.61),
        ("City 5",  272445.06,  18665211, 69, 61, 120.67, 68.96,   383, 8.92,  11.21, 70.0,  7.0, 5.0, 22,  98.37),
        ("City 6",  893992.89,  63312576, 71, 60,  96.50, 65.89,  1645, 7.83,   7.09, 68.0,  7.0, 8.0, 27, 111.73),
        ("City 7",  283652.73,  21407272, 75, 67,  88.88, 48.23,   774, 7.93,   6.16, 80.0, 15.0, 5.0, 32,  92.83),
        ("City 8",    6443.55,    356264, 55, 49, 101.95, 65.52,    11, 7.90,   2.23, 78.0, 11.0, 4.0, 35, 102.19),
        ("City 9",  901869.45,  60758945, 67, 60, 165.28, 41.49,  1539, 7.50,   5.90, 78.0, 10.0, 4.0, 25, 104.94),
    ]
    cols = [
        "cidade", "orders", "gmv", "aov", "aov_basket", "saturacao_pct", "ocupacao_pct",
        "entregadores", "margem_contribuicao_pct", "margem_base_pct", "share_pct",
        "custo_hora_incremental", "custo_pedido_incremental", "frete_gratis_pct", "price_index_pct"
    ]
    df = pd.DataFrame(raw_data, columns=cols)
    for c in candidates:
        if os.path.exists(c):
            print("Fonte localizada:", c)
            break
    df["pedidos_base_k"] = df["orders"] / 1000.0
    df["gmv_base_m"] = df["gmv"] / 1e6
    df["margem_contribuicao_rs"] = df["gmv"] * (df["margem_contribuicao_pct"] / 100.0)
    df["margem_contribuicao_m"] = df["margem_contribuicao_rs"] / 1e6
    df["operating_margin_rs"] = df["gmv"] * (df["margem_base_pct"] / 100.0)
    df["margem_base_m"] = df["operating_margin_rs"] / 1e6
    df["capacidade_base_pedidos_equiv"] = df["orders"] / (df["saturacao_pct"] / 100.0)
    df["criterio_materialidade"] = np.where(
        df["gmv"] >= 50_000_000,
        "Atende (GMV >= R$ 50M)",
        np.where(
            df["cidade"] == "City 18",
            "Excecao Estrategica (Folga logistica + Share 65% + Retorno marginal positivo)",
            "Fora de Escopo (GMV < R$ 50M)"
        )
    )
    df.to_csv("data/processed/base_validada.csv", index=False)
    return df

df_base = load_or_build_base()

alloc_rows = [
    {
        "cidade": "City 3",
        "classificacao": "Ataque",
        "alavanca": "Incentivos para clientes",
        "custo_bruto_rs": 2750000.00,
        "pedidos_brutos": 121000.00,
        "canibalizacao_pct": 18.0,
        "pedidos_incrementais_liquidos": 99220.00,
        "gmv_incremental_rs": 7639940.00,
        "mc_incremental_rs": 628003.07,
        "custo_operacional_sla_rs": 192500.00,
        "impacto_liquido_om_rs": -2314496.93,
        "mc_por_real_custo": 0.2284,
        "multiplo_gmv": 2.78,
        "saturacao_base_pct": 79.81,
        "saturacao_pos_investimento_pct": 83.20,
        "capacidade_base_pedidos_equiv": round(1318842.06 / 0.7981, 2),
        "capacidade_adicional_pedidos_equiv": 0.00,
        "capacidade_adicional_pct": 0.00,
        "saturacao_sem_intervencao_pct": round((1318842.06 + 99220.00) / (1318842.06 / 0.7981) * 100.0, 2),
        "folga_guardrail_98_pp": 14.80,
        "justificativa": "Folga logistica (79,81%), OM saudavel (6,06%), GMV relevante (R$ 101,21M) e maior retorno marginal em MC/R$ (R$ 0,2284)."
    },
    {
        "cidade": "City 2",
        "classificacao": "Ataque",
        "alavanca": "Alteracoes de preco",
        "custo_bruto_rs": 2250000.00,
        "pedidos_brutos": 81000.00,
        "canibalizacao_pct": 12.0,
        "pedidos_incrementais_liquidos": 71280.00,
        "gmv_incremental_rs": 5132160.00,
        "mc_incremental_rs": 459841.54,
        "custo_operacional_sla_rs": 135000.00,
        "impacto_liquido_om_rs": -1925158.46,
        "mc_por_real_custo": 0.2044,
        "multiplo_gmv": 2.28,
        "saturacao_base_pct": 83.86,
        "saturacao_pos_investimento_pct": 86.50,
        "capacidade_base_pedidos_equiv": round(1213681.77 / 0.8386, 2),
        "capacidade_adicional_pedidos_equiv": 0.00,
        "capacidade_adicional_pct": 0.00,
        "saturacao_sem_intervencao_pct": round((1213681.77 + 71280.00) / (1213681.77 / 0.8386) * 100.0, 2),
        "folga_guardrail_98_pp": 11.50,
        "justificativa": "Folga logistica (83,86%), OM alta (8,41%), share de 67% e Price Index de 116,84% com canibalizacao contida (12%)."
    },
    {
        "cidade": "City 18",
        "classificacao": "Ataque (Excecao de Share)",
        "alavanca": "Incentivos para clientes",
        "custo_bruto_rs": 1750000.00,
        "pedidos_brutos": 77000.00,
        "canibalizacao_pct": 18.0,
        "pedidos_incrementais_liquidos": 63140.00,
        "gmv_incremental_rs": 4167240.00,
        "mc_incremental_rs": 321710.93,
        "custo_operacional_sla_rs": 122500.00,
        "impacto_liquido_om_rs": -1550789.07,
        "mc_por_real_custo": 0.1838,
        "multiplo_gmv": 2.38,
        "saturacao_base_pct": 78.48,
        "saturacao_pos_investimento_pct": 82.98,
        "capacidade_base_pedidos_equiv": round(630705.33 / 0.7848, 2),
        "capacidade_adicional_pedidos_equiv": 0.00,
        "capacidade_adicional_pct": 0.00,
        "saturacao_sem_intervencao_pct": round((630705.33 + 63140.00) / (630705.33 / 0.7848) * 100.0, 2),
        "folga_guardrail_98_pp": 15.02,
        "justificativa": "Excecao estrategica de ataque de share em praca com folga logistica (78,48%), menor share da base (65%) e retorno marginal positivo (R$ 0,1838 MC/R$)."
    },
    {
        "cidade": "City 6",
        "classificacao": "Ataque",
        "alavanca": "Frete gratis",
        "custo_bruto_rs": 1700000.00,
        "pedidos_brutos": 64600.00,
        "canibalizacao_pct": 22.0,
        "pedidos_incrementais_liquidos": 50388.00,
        "gmv_incremental_rs": 3577548.00,
        "mc_incremental_rs": 280122.01,
        "custo_operacional_sla_rs": 161000.00,
        "impacto_liquido_om_rs": -1580877.99,
        "mc_por_real_custo": 0.1648,
        "multiplo_gmv": 2.10,
        "saturacao_base_pct": 96.50,
        "saturacao_pos_investimento_pct": 97.54,
        "capacidade_base_pedidos_equiv": round(893992.89 / 0.9650, 2),
        "capacidade_adicional_pedidos_equiv": 0.00,
        "capacidade_adicional_pct": 0.00,
        "saturacao_sem_intervencao_pct": 97.54,
        "folga_guardrail_98_pp": 0.46,
        "justificativa": "Saturacao base 96,50%, pos-investimento 97,54% (<98%), folga remanescente de 0,46 p.p., sem aumento de capacidade; custo op./SLA de R$ 161.000,00 (8% custo bruto + R$ 25.000,00 SLA extra)."
    },
    {
        "cidade": "City 17",
        "classificacao": "Capacidade / Turnaround",
        "alavanca": "Capacidade logistica (entregadores)",
        "custo_bruto_rs": 2665969.51,
        "pedidos_brutos": 74647.15,
        "canibalizacao_pct": 8.0,
        "pedidos_incrementais_liquidos": 68675.38,
        "gmv_incremental_rs": 5562705.78,
        "mc_incremental_rs": 410527.69,
        "custo_operacional_sla_rs": 373235.73,
        "impacto_liquido_om_rs": -2628677.55,
        "mc_por_real_custo": 0.1540,
        "multiplo_gmv": 2.09,
        "saturacao_base_pct": 131.38,
        "saturacao_pos_investimento_pct": 121.54,
        "capacidade_base_pedidos_equiv": 1524881.30,
        "capacidade_adicional_pedidos_equiv": 180000.00,
        "capacidade_adicional_pct": 11.80,
        "saturacao_sem_intervencao_pct": 135.88,
        "folga_guardrail_98_pp": -23.54,
        "justificativa": "Praca estrutural (GMV R$ 162,62M, sat. 131,38%, OM -1,34%); +180.000 pedidos equiv. (+11,80% cap. base, 296.218,83 h a R$ 9/h) reduz sat. de 135,88% s/ interv. para 121,54%, sem cupom generico."
    }
]
df_alloc = pd.DataFrame(alloc_rows)
df_alloc.to_csv("data/processed/base_alocacao_otimizada.csv", index=False)

# Auditoria de AOV e MC contra base real
m = df_alloc.merge(df_base[["cidade", "aov", "margem_contribuicao_pct", "share_pct", "saturacao_pct", "margem_base_pct", "price_index_pct"]], on="cidade")
m["aov_modelo"] = (m["gmv_incremental_rs"] / m["pedidos_incrementais_liquidos"]).round(2)
m["mc_pct_modelo"] = (m["mc_incremental_rs"] / m["gmv_incremental_rs"] * 100.0).round(2)
m["diff_aov"] = (m["aov_modelo"] - m["aov"]).round(2)
m["diff_mc_pp"] = (m["mc_pct_modelo"] - m["margem_contribuicao_pct"]).round(2)
display(m[["cidade", "aov", "aov_modelo", "diff_aov", "margem_contribuicao_pct", "mc_pct_modelo", "diff_mc_pp", "share_pct", "saturacao_pct", "price_index_pct"]])
